# A10 · Gemma 4 (sliding/global · K=V · PLE · KV 공유 · MoE) 학습 동작 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/train_verify/A10_gemma4.ipynb)

원본: transformers `gemma4` 의 `Gemma4TextModel` 과 `gemma4_unified` 의 `Gemma4UnifiedTextModel`
(tag `v5.20.0` = commit `48e4bcd7`; 2026-10 `main`(`683b7219`) 의 두 폴더 파일과 글자 단위로 같음).
내 구현: `miniarch/models/gemma4.py`. 텍스트 backbone 만 비교합니다 (vision/audio 인코더 제외).

체크포인트 (config.json 확인): 31B (dense, K=V), 26B-A4B (K=V + dense MLP·MoE 병렬, expert 128 중 8),
E4B / E2B (PLE + 마지막 18/20 층 KV 공유, E2B 는 공유 층 MLP 2배 폭), 12B (`gemma4_unified`, 2026-05).

부품
- sliding 5 : global 1 하이브리드. sliding 층 head_dim 256 / global 층 512, global 층 KV head 수 따로 (`num_global_key_value_heads`)
- QK-norm + V-norm(가중치 없음), softmax 스케일 1.0
- `attention_k_eq_v`: global 층은 v_proj 없이 k_proj 출력을 V 로 사용 (K 는 k_norm+RoPE, V 는 V-norm)
- RoPE 층 종류별: sliding 은 기본(θ=1e4), global 은 proportional(θ=1e6, 주파수 앞 25% 만 회전)
- Gemma sandwich norm (attention·FFN 앞뒤), 임베딩 × √d, 층 끝 `layer_scalar`, GeGLU(gelu_tanh) MLP, logit softcapping(30)
- MoE (26B-A4B): dense MLP 와 expert 가지를 둘 다 계산해 각각 norm 후 더함. router 는 norm·scale·per_expert_scale
- PLE (E 계열): 층마다 토큰별 작은 벡터를 게이트에 곱해 residual 에 주입
- KV 공유 (E 계열): 마지막 층들은 공유 직전 같은 종류 층의 K/V 를 재사용, 2배 폭 MLP 스위치

`gemma4_unified` 텍스트 모델은 PLE·MoE 항목이 빠졌을 뿐 attention·MLP·RoPE·층 계산이 gemma4 와 같은 코드입니다
(생성된 modeling 파일 diff 로 확인). 별도 스위치 없이 같은 구현으로 자체 점검에서 한 번 더 비교합니다.

축소 config (본 비교): 7층 = [S, F, S, S, F, S, F], window 5, 시퀀스 24 — 원본이 허용하는 기능을 한꺼번에 켬
- head_dim sliding 16 / global 32, KV head sliding 2 / global 1, K=V 켬 → 설정이 뒤바뀌면 shape 부터 어긋남
- global RoPE: 32 × 0.25 / 2 = 주파수 4개만 회전
- 마지막 3층 KV 공유 (F, S, F 가 각각 layer 1, 3 의 K/V 사용) + 공유 층 2배 폭 MLP
- MoE expert 8개 중 3개 + dense MLP, PLE dim 8
- 원본 초기값이 1 인 `layer_scalar`, `per_expert_scale` 과 작은 router 가중치는 0/1 이 아닌 값으로 채움

자체 점검: E 계열(K=V 끔), dense 31B 형, `Gemma4UnifiedTextModel`, `Gemma4ForCausalLM`(softcapping 된 logits) 각각 원본과 비교.

순서: 0. 환경 준비 → 1. 기울기 == 원본 → 2. 학습 궤적 == 원본 → 3. 배치 하나 외우기 → 4. float64 수치 미분 → 5. 라우터 bias 업데이트 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
if not os.environ.get('MINIARCH_SKIP_PIP'):   # 전체 실행 노트북이 한 번에 설치
    !pip -q install -U 'transformers>=5.20.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A10")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

### 원본과 비교할 수 없는 부분

- **수치 미분 (float64)**: 무작위 방향으로 가중치를 ±ε 움직여 잰 손실 변화가 autograd 기울기와 같은지 봅니다.
  원본이 추론 전용이라 기울기를 비교할 수 없는 모델도 이걸로 backward 를 확인합니다.
- **라우터 bias 업데이트 (aux-loss-free)**: 원본 추론 코드에는 없는 학습 전용 규칙입니다. 많이 뽑힌 expert 의 bias 는 내려가고
  적게 뽑힌 쪽은 올라가는지, bias 가 optimizer 에서 빠져 있는지, bias 가 실제로 선택을 바꾸는지 봅니다. 라우터 bias 가 없는 모델은 건너뜁니다.

In [ ]:
try:
    f = TR.directional_grad_check(pair["mine"], pair, inputs, dev)
    print(f"방향 {len(f.rel_errors)}개 상대 오차 {[f'{e:.1e}' for e in f.rel_errors]}  (ε={f.eps})")
    report("수치 미분 == autograd (float64)", f.ok, max(f.rel_errors))
except Exception:
    traceback.print_exc(limit=4); report("수치 미분 == autograd (float64)", False)
try:
    b = TR.router_bias_check(pair["mine"], pair, inputs, dev)
    print("\n".join(b.details))
    report("라우터 bias 업데이트" if b.n_routers else "라우터 bias 업데이트 (라우터 bias 없음 → 건너뜀)", b.ok)
except Exception:
    traceback.print_exc(limit=4); report("라우터 bias 업데이트", False)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/train_verify/A10_gemma4.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/train_A10_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")